# 02 · Build the team's Kaggle Dataset `galaxy10-decals-processed-v1`  (issue #10)

Run this **once** (by one person). It produces a clean `/kaggle/working` that becomes a private Kaggle Dataset. After
that, every teammate attaches the dataset to their notebooks and skips the 2.7 GB download and the cache build.

**Settings:** Accelerator *None* (CPU is enough) · Internet *On*. Then *Save Version → Save & Run All (Commit)*.
When the version has finished, open it → **Output** tab → **New Dataset** (see `docs/KAGGLE_DATASET.md`).

Output layout (about 3.5 GB):
```
processed/images_256.npy   uint8 (17736, 256, 256, 3), rows = orig_index (all images, incl. the 118 removed ones)
processed/meta.csv         split_v1 metadata
splits/                    split_v1.csv, band_stats_v1.json, removed_v1.csv, dedup_report_v1.json (copies of git)
README.md, SHA256SUMS.txt
```

In [ ]:
import os, subprocess
WORK = "/kaggle/working"                     # 20 GB limit; peak use here ≈ 6.5 GB, final output ≈ 3.5 GB
REPO_DIR = f"{WORK}/CS464_Project"
DATA_DIR = f"{WORK}/data"
os.environ["GALAXY_DATA_DIR"] = DATA_DIR
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", "-b", "main", "https://github.com/ErenKerimoglu/CS464_Project.git", REPO_DIR],
                   check=True)
%cd {REPO_DIR}
!git log --oneline -1
!pip install -q -e .
!df -h {WORK}

## Build (download → verify → reproduce split → cache)

In [ ]:
!python scripts/prepare_data.py

## Assemble the clean output

In [ ]:
import hashlib, json, shutil, datetime
import numpy as np
from pathlib import Path

out = Path(WORK)
%cd {WORK}
commit = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
(out / "processed").mkdir(exist_ok=True)
shutil.move(f"{DATA_DIR}/processed/images_256.npy", out / "processed/images_256.npy")
shutil.copy(f"{DATA_DIR}/processed/meta.csv", out / "processed/meta.csv")
shutil.copytree(f"{REPO_DIR}/data/splits", out / "splits", ignore=shutil.ignore_patterns(".gitkeep"), dirs_exist_ok=True)
for p in out.iterdir():                      # keep only the dataset files (drops the raw h5 and the repo clone)
    if p.name not in {"processed", "splits"}:
        shutil.rmtree(p) if p.is_dir() else p.unlink()

def sha256(p, chunk=1 << 24):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        while b := f.read(chunk):
            h.update(b)
    return h.hexdigest()

files = sorted(p for p in out.rglob("*") if p.is_file())
sums = {str(p.relative_to(out)): sha256(p) for p in files}
(out / "SHA256SUMS.txt").write_text("".join(f"{h}  {n}\n" for n, h in sums.items()))
arr = np.load(out / "processed/images_256.npy", mmap_mode="r")
(out / "README.md").write_text(f'''# Galaxy10 DECaLS, processed (CS464 Group 10), v1

Built by `notebooks/02_build_kaggle_dataset.ipynb` from https://github.com/ErenKerimoglu/CS464_Project (commit {commit})
on {datetime.date.today()}. Source: astroNN Galaxy10 DECaLS, Zenodo record 10845026 (MD5-verified).

* `processed/images_256.npy`: uint8 {arr.shape}, row i = orig_index i (all original images; the split excludes 118
  label-conflict duplicates).
* `processed/meta.csv` + `splits/`: the frozen split_v1 and train band statistics (identical to git).
* Attach this dataset to a Kaggle notebook: `galaxy.constants` finds `processed/` automatically.
''')
print(arr.shape, arr.dtype)
for n, h in sums.items():
    print(f"{h[:16]}…  {n}")
!du -sh /kaggle/working/* 2>/dev/null